In [30]:
import pandas as pd

file_path = "../RawData/NewsData14200records.csv"
data = pd.read_csv(file_path)


In [31]:
print("\nข้อมูลสรุป:")
print(data.info()) 


ข้อมูลสรุป:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14321 entries, 0 to 14320
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Public_Date_Time   14321 non-null  object
 1   URL                14321 non-null  object
 2   Title              14321 non-null  object
 3   Body               14321 non-null  object
 4   Verify_Department  12251 non-null  object
 5   Types              14318 non-null  object
 6   category           14321 non-null  object
 7   Viewers            14321 non-null  int64 
 8   Hashtag            14321 non-null  object
dtypes: int64(1), object(8)
memory usage: 1007.1+ KB
None


In [20]:
print("\nจำนวนข้อมูลที่หายไปในแต่ละคอลัมน์:")
print(data.isnull().sum())


จำนวนข้อมูลที่หายไปในแต่ละคอลัมน์:
Public_Date_Time        0
URL                     0
Title                   0
Body                    0
Verify_Department    2070
Types                   3
category                0
Viewers                 0
Hashtag                 0
dtype: int64


In [4]:
print("\nสถิติพื้นฐาน:")
print(data.describe())


สถิติพื้นฐาน:
             Viewers
count   14321.000000
mean     1548.408910
std      7952.553743
min         4.000000
25%       133.000000
50%       305.000000
75%      1151.000000
max    499633.000000


In [6]:
print("\nค่าที่ไม่ซ้ำในคอลัมน์ 'Types':")
print(data['Types'].unique())


ค่าที่ไม่ซ้ำในคอลัมน์ 'Types':
['ข่าวปลอม' 'ข่าวจริง' 'คลังความรู้' 'อาชญากรรมออนไลน์' 'ข่าวบิดเบือน'
 'ข่าวอื่นๆ' 'กิจกรรม' 'ข่าวสาร' 'นโยบายรัฐบาล-ข่าวสาร' nan 'การเงิน-หุ้น'
 'ผลิตภัณฑ์สุขภาพ' 'ยาเสพติด']


In [7]:
# ลบคอลัมน์ 'Verify_Department'
data = data.drop(columns=['Verify_Department','Public_Date_Time', 'URL'])


In [32]:
data_clean = data.dropna()

In [22]:
print(data_clean.info())

<class 'pandas.core.frame.DataFrame'>
Index: 12251 entries, 0 to 14221
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Public_Date_Time   12251 non-null  object
 1   URL                12251 non-null  object
 2   Title              12251 non-null  object
 3   Body               12251 non-null  object
 4   Verify_Department  12251 non-null  object
 5   Types              12251 non-null  object
 6   category           12251 non-null  object
 7   Viewers            12251 non-null  int64 
 8   Hashtag            12251 non-null  object
dtypes: int64(1), object(8)
memory usage: 957.1+ KB
None


In [33]:
data_clean.loc[:, 'Types'] = data_clean['Types'].astype('category')
print(data_clean['Types'].isna().sum())  # ตรวจสอบจำนวน NaN
print(data_clean['Types'].unique()) 

0
['ข่าวปลอม' 'ข่าวจริง' 'คลังความรู้' 'อาชญากรรมออนไลน์' 'ข่าวบิดเบือน'
 'ข่าวอื่นๆ' 'กิจกรรม' 'ข่าวสาร' 'นโยบายรัฐบาล-ข่าวสาร' 'ผลิตภัณฑ์สุขภาพ']


In [34]:
data_clean.loc[:, 'Types'] = data_clean['Types'].str.strip()

In [27]:
# กรองข้อมูลให้เหลือแค่ 'ข่าวจริง' และ 'ข่าวปลอม' เท่านั้น
df_filtered = data_clean[data_clean['Types'].isin(['ข่าวจริง', 'ข่าวปลอม'])]

# ตรวจสอบจำนวนของแต่ละประเภทหลังการกรอง
print("\nจำนวนของแต่ละประเภทในคอลัมน์ 'Types' หลังการกรอง:")
print(df_filtered['Types'].value_counts())


จำนวนของแต่ละประเภทในคอลัมน์ 'Types' หลังการกรอง:
Types
ข่าวปลอม    6780
ข่าวจริง    2390
Name: count, dtype: int64


In [13]:
# ดุลข่าวแต่ละประเภท ลดข่าวปลอมลง
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

print("จำนวนข้อมูลในแต่ละคลาสก่อนทำ Oversampling/Undersampling:")
print(Counter(df_filtered['Types']))

X = df_filtered.drop(columns=['Types'])  # Features
y = df_filtered['Types']  # Target

undersample = RandomUnderSampler(sampling_strategy={'ข่าวปลอม': 2718}, random_state=42)
X_under, y_under = undersample.fit_resample(X, y)

print("จำนวนข้อมูลหลังทำ Undersampling:", Counter(y_under))

จำนวนข้อมูลในแต่ละคลาสก่อนทำ Oversampling/Undersampling:
Counter({'ข่าวปลอม': 8182, 'ข่าวจริง': 2718})
จำนวนข้อมูลหลังทำ Undersampling: Counter({'ข่าวจริง': 2718, 'ข่าวปลอม': 2718})


In [26]:
print(X_under.info())  # ดู 5 แถวแรกของ features
print(pd.Series(y_under).value_counts())

<class 'pandas.core.frame.DataFrame'>
Index: 5436 entries, 1 to 9676
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Title     5436 non-null   object
 1   Body      5436 non-null   object
 2   category  5436 non-null   object
 3   Viewers   5436 non-null   int64 
 4   Hashtag   5436 non-null   object
dtypes: int64(1), object(4)
memory usage: 254.8+ KB
None
0    2718
1    2718
Name: count, dtype: int64


In [15]:
print(X_under.info())
print(y_under.count())

<class 'pandas.core.frame.DataFrame'>
Index: 5436 entries, 1 to 9676
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Title     5436 non-null   object
 1   Body      5436 non-null   object
 2   category  5436 non-null   object
 3   Viewers   5436 non-null   int64 
 4   Hashtag   5436 non-null   object
dtypes: int64(1), object(4)
memory usage: 254.8+ KB
None
5436


In [16]:
print(y_under)

0       ข่าวจริง
2       ข่าวจริง
4       ข่าวจริง
6       ข่าวจริง
11      ข่าวจริง
          ...   
4205    ข่าวปลอม
2126    ข่าวปลอม
2810    ข่าวปลอม
4256    ข่าวปลอม
1875    ข่าวปลอม
Name: Types, Length: 4245, dtype: object


In [16]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_under = label_encoder.fit_transform(y_under)

# แสดงผลลัพธ์ที่แปลง 0 1 แล้ว
print(y_under)

[0 0 0 ... 1 1 1]


Save Result

In [ ]:
import joblib
# บันทึก y_under
joblib.dump(X_under, 'result/X_under.pkl')
joblib.dump(y_under, 'result/y_under.pkl')

print("✅ บันทึกข้อมูลเรียบร้อยแล้ว!")


✅ บันทึกข้อมูลเรียบร้อยแล้ว!
